# Phase 16: PF-X2 — corrected engine + gated confidence hedge

Carries in notebook 15's fleet verdicts:
- **GR-mixing retired** — falsified in all three forms (causal wrong-side, causal
  both-sides, centered both-sides); the code path is removed, not defaulted off.
- **dz-likelihood + likelihood-weighted seeds kept** — both validated at fleet
  scale (−0.71 and −0.76 in the ablation). This notebook runs their combination
  at full strength for the first time (nb15's headline had the dead mechanism
  mixed in).
- **Projection kept** (fleet verdict: helps, ~70% of wells).

New: the **seed-spread confidence hedge**, targeting the 28 disaster wells that
carry 53% of pooled error — but as a **gated hypothesis**. The dev well showed
row-level spread can anti-correlate with error (consensus failures), and we have
precedent for confidence gates failing (`z_span` AUC 0.51, nb7). So: a fleet
diagnostic measures several candidate signals against per-well error first; the
hedge is tuned (out-of-fold) only if a signal clears the gate, and the notebook
reports "hedge off" honestly if none does.

Same seeded 250-well sample as nb15 → tables directly comparable
(nb15 refs on these wells: floor 18.009 / pfx 15.172 / pfx_proj 15.020 pooled).

## Config

In [1]:
import warnings; warnings.filterwarnings("ignore")
import time
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr

TRAIN_DIR = Path("../data/raw/train")

WELL_SAMPLE = 250
ABLATION_GATE_RHO = 0.30      # min |spearman| for a confidence signal to enable the hedge
N_FOLDS = 5
EVAL_FRAC = 0.73

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

print(f"train dir exists: {TRAIN_DIR.exists()}")

train dir exists: True


## Engine v2 (GR path removed; returns pred, spread, aux)

In [2]:
def load_raw_well(hz_path, tw_path):
    h = pd.read_csv(hz_path).sort_values("MD").reset_index(drop=True)
    t = pd.read_csv(tw_path)
    tw = t.dropna(subset=["TVT", "GR"]).sort_values("TVT")
    twt = tw["TVT"].values.astype(float)
    twg = tw["GR"].clip(0, 300).values.astype(float)
    Z = h["Z"].values.astype(float); MD = h["MD"].values.astype(float)
    gr = pd.Series(h["GR"].clip(0, 300).values).interpolate(
        limit_direction="both").fillna(90.0).values
    return h, twt, twg, Z, MD, gr

def pfx2(tvt_known, Z, MD, gr, twt, twg, kn, ev,
         S=16, N=500, mom=0.998, vn=0.002, pn=0.005,
         dz_lik=True, zsig_mult=2.0, lik_weight=True, temp=1.0, seed=42):
    # GR-mixing retired (falsified in causal, both-sides and centered forms).
    # Returns (pred, spread, aux): spread = per-row lik-weighted std across seeds;
    # aux carries per-well confidence candidates for the hedge diagnostic.
    rng = np.random.default_rng(seed)
    last = kn[-1]
    gs = float(np.clip(np.nanstd(gr[kn] - np.interp(tvt_known[kn], twt, twg)), 10.0, 60.0))
    dzk = np.diff(Z[kn]); dvt = np.diff(tvt_known[kn]); dmk = np.diff(MD[kn]); m = dmk > 0
    if m.sum() >= 10:
        vz = dzk[m] / dmk[m]; vt = dvt[m] / dmk[m]
        A = np.column_stack([vz, np.ones_like(vz)])
        c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt = float(c[0]), float(c[1])
        zsig = max(float(np.std(vt - (c[0] * vz + c[1]))), 1e-3)
    else:
        beta, icpt, zsig = -1.0, 0.0, 0.1
    tl = kn[-20:]; d2 = np.diff(tvt_known[tl]); dm2 = np.diff(MD[tl]); mm = dm2 > 0
    iv = float(np.median(d2[mm] / dm2[mm])) if mm.sum() >= 3 else 0.0
    lo, hi = twt[0] - 50.0, twt[-1] + 50.0
    pos = tvt_known[last] + 0.5 * rng.standard_normal((S, N))
    vel = iv + 0.02 * rng.standard_normal((S, N))
    w = np.ones((S, N)) / N
    LL = np.zeros(S)
    out = np.empty((S, len(ev)))
    prev_md, prev_z = MD[last], Z[last]
    for i, idx in enumerate(ev):
        dm = max(MD[idx] - prev_md, 1.0); dzd = (Z[idx] - prev_z) / dm
        vel = mom * vel + vn * rng.standard_normal((S, N))
        pos = np.clip(pos + vel * dm + pn * rng.standard_normal((S, N)), lo, hi)
        g = gr[idx]
        if np.isfinite(g):
            eg = np.interp(pos.ravel(), twt, twg).reshape(S, N)
            d2r = ((g - eg) / gs) ** 2
            w = w * np.maximum(np.exp(-0.5 * np.minimum(d2r, 600.0)), 1e-300)
        if dz_lik:
            ve = beta * dzd + icpt
            dv = (vel - ve) / max(zsig * zsig_mult, 5e-3)
            w = w * np.maximum(np.exp(-0.5 * np.minimum(dv * dv, 600.0)), 1e-300)
        ws = w.sum(1, keepdims=True)
        LL += np.log(np.maximum(ws.ravel() / N, 1e-300))
        w = np.where(ws > 0, w / np.maximum(ws, 1e-300), 1.0 / N)
        ess = 1.0 / np.maximum((w * w).sum(1), 1e-300)
        for s in np.where(ess < 0.5 * N)[0]:
            ci = np.clip(np.searchsorted(np.cumsum(w[s]),
                 (np.arange(N) + rng.uniform(0, 1)) / N), 0, N - 1)
            pos[s] = pos[s][ci] + 0.1 * rng.standard_normal(N)
            vel[s] = vel[s][ci] + 0.003 * rng.standard_normal(N)
            w[s] = 1.0 / N
        out[:, i] = (w * pos).sum(1)
        prev_md, prev_z = MD[idx], Z[idx]
    if lik_weight and S > 1:
        LLn = (LL - LL.max()) / max(temp, 1e-6)
        sw = np.exp(LLn); sw /= sw.sum()
        pred = (out * sw[:, None]).sum(0)
        spread = np.sqrt(np.maximum((sw[:, None] * (out - pred) ** 2).sum(0), 0.0))
    else:
        sw = np.full(S, 1.0 / S)
        pred = out.mean(0)
        spread = out.std(0)
    uni = out.mean(0)
    aux = dict(
        gs=gs,
        spread_mean=float(spread.mean()),
        spread_p90=float(np.quantile(spread, 0.90)),
        end_spread=float(np.sqrt(max((sw * (out[:, -1] - pred[-1]) ** 2).sum(), 0.0))),
        lw_shift=float(np.mean(np.abs(pred - uni))),
    )
    return pred, spread, aux

def project(pred, Z_ev, MD_ev, anchor_tvt, anchor_z, anchor_md, deg=4, blend=0.75):
    u = pred + Z_ev - (anchor_tvt + anchor_z)
    span = MD_ev[-1] - anchor_md
    if len(pred) < deg + 3 or span <= 0:
        return pred
    s = (MD_ev - anchor_md) / span
    c = np.polyfit(s, u, deg)
    for _ in range(4):
        r = u - np.polyval(c, s)
        sc = np.median(np.abs(r)) * 1.4826 + 1e-6
        c = np.polyfit(s, u, deg, w=1.0 / (1.0 + (r / (2.0 * sc)) ** 2))
    u2 = blend * np.polyval(c, s) + (1 - blend) * u
    return u2 + (anchor_tvt + anchor_z) - Z_ev

## Fleet CV — the corrected config at full strength

References on the SAME wells (nb15): floor 18.009 / 13.873 · pfx 15.172 / 11.525
· pfx_proj 15.020 / 11.298. Ablation arithmetic predicted ~10.0–10.4 per-well
for this config.

In [3]:
files = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
wids = [f.name.split("__")[0] for f in files
        if (TRAIN_DIR / f"{f.name.split('__')[0]}__typewell.csv").exists()]
if WELL_SAMPLE and WELL_SAMPLE < len(wids):
    wids = sorted(np.random.default_rng(123).choice(wids, WELL_SAMPLE, replace=False))
print(f"CV on {len(wids)} wells")

store = {}
rows = []; t0 = time.time()
for k, wid in enumerate(wids):
    try:
        h, twt, twg, Z, MD, gr = load_raw_well(
            TRAIN_DIR / f"{wid}__horizontal_well.csv",
            TRAIN_DIR / f"{wid}__typewell.csv")
    except Exception:
        continue
    if "TVT" not in h or h["TVT"].isna().all() or len(twt) < 3:
        continue
    tvt = h["TVT"].values.astype(float); n = len(h)
    kcut = int(round(n * EVAL_FRAC))
    if kcut < 20 or n - kcut < 20:
        continue
    kn = np.arange(0, n - kcut); ev = np.arange(n - kcut, n)
    true = tvt[ev]; anchor = float(tvt[kn[-1]])
    pred, spread, aux = pfx2(tvt, Z, MD, gr, twt, twg, kn, ev)
    pj = project(pred, Z[ev], MD[ev], tvt[kn[-1]], Z[kn[-1]], MD[kn[-1]])
    store[wid] = dict(pred=pj, spread=spread, true=true, anchor=anchor,
                      n_eval=len(ev))
    rows.append(dict(well=wid, n_eval=len(ev), floor=rmse(anchor, true),
                     pfx2=rmse(pred, true), pfx2_proj=rmse(pj, true), **aux))
    if (k + 1) % 25 == 0:
        print(f"  ...{k+1}/{len(wids)} [{(time.time()-t0)/60:.1f} min]")

cv = pd.DataFrame(rows)
cv.to_csv("../data/interim/pfx2_cv.csv", index=False)

def pooled(col):
    return float(np.sqrt((cv[col] ** 2 * cv["n_eval"]).sum() / cv["n_eval"].sum()))
print(f"\n{'model':12s}{'pooled':>9s}{'per-well':>10s}")
for c in ["floor", "pfx2", "pfx2_proj"]:
    print(f"{c:12s}{pooled(c):9.3f}{cv[c].mean():10.3f}")
print("(nb15 same wells: floor 18.009/13.873 | pfx 15.172/11.525 | "
      "pfx_proj 15.020/11.298)")

CV on 250 wells
  ...25/250 [1.0 min]
  ...50/250 [1.9 min]
  ...75/250 [2.9 min]
  ...100/250 [3.8 min]
  ...125/250 [4.8 min]
  ...150/250 [5.7 min]
  ...175/250 [6.7 min]
  ...200/250 [7.7 min]
  ...225/250 [8.6 min]
  ...250/250 [9.5 min]

model          pooled  per-well
floor          18.009    13.873
pfx2           15.023    11.453
pfx2_proj      14.854    11.209
(nb15 same wells: floor 18.009/13.873 | pfx 15.172/11.525 | pfx_proj 15.020/11.298)


## Confidence-signal diagnostic (the gate)

Does any observable separate the disaster wells? Spearman of each candidate
against per-well error, plus disaster-cohort medians. The hedge only proceeds
if the best signal clears `ABLATION_GATE_RHO`. Precedent says this can fail
(`z_span` AUC 0.51) — failing honestly here is a valid outcome.

In [4]:
SIGNALS = ["spread_mean", "spread_p90", "end_spread", "lw_shift", "gs"]
tgt = cv["pfx2_proj"]
print(f"{'signal':12s}{'spearman':>10s}")
rhos = {}
for s in SIGNALS:
    r = spearmanr(cv[s], tgt).statistic
    rhos[s] = 0.0 if not np.isfinite(r) else float(r)
    print(f"{s:12s}{rhos[s]:10.3f}")

dis = cv[cv["pfx2_proj"] > 2 * cv["floor"]]
print(f"\ndisaster wells (err > 2x floor): {len(dis)}")
if len(dis):
    for s in SIGNALS:
        print(f"  {s:12s} disaster median {dis[s].median():8.3f} | "
              f"rest median {cv.loc[~cv.index.isin(dis.index), s].median():8.3f}")

BEST_SIG = max(rhos, key=lambda k: abs(rhos[k]))
GATE_OPEN = abs(rhos[BEST_SIG]) >= ABLATION_GATE_RHO
print(f"\nbest signal: {BEST_SIG} (rho={rhos[BEST_SIG]:+.3f}) -> "
      f"hedge {'ENABLED' if GATE_OPEN else 'DISABLED (signal too weak — honest no-op)'}")

signal        spearman
spread_mean     -0.248
spread_p90      -0.247
end_spread      -0.250
lw_shift         0.125
gs               0.135

disaster wells (err > 2x floor): 21
  spread_mean  disaster median    0.009 | rest median    0.021
  spread_p90   disaster median    0.019 | rest median    0.044
  end_spread   disaster median    0.003 | rest median    0.007
  lw_shift     disaster median    1.438 | rest median    1.662
  gs           disaster median   13.816 | rest median   13.221

best signal: end_spread (rho=-0.250) -> hedge DISABLED (signal too weak — honest no-op)


## Hedge (only if the gate opened) — out-of-fold tuned

Per-well hedge: wells whose signal exceeds a train-fold quantile get their whole
tail blended toward hold. Parameters `(quantile, weight)` picked per fold on
training wells, applied to validation wells. No-op is always in the grid.

In [5]:
HEDGED = {w: store[w]["pred"] for w in store}      # default: unhedged
hedge_result = None
if GATE_OPEN:
    sign = np.sign(rhos[BEST_SIG]) or 1.0          # hedge the high-error side
    sig = {w: sign * cv.set_index("well").loc[w, BEST_SIG] for w in store}
    wells_l = sorted(store)
    fold_of = {w: i for i, w in zip(
        np.random.default_rng(0).permutation(len(wells_l)) % N_FOLDS, wells_l)}
    GRID = [(None, 0.0)] + [(q, wgt) for q in (0.70, 0.85) for wgt in (0.3, 0.5, 0.8)]

    def pooled_pv(pv, ws):
        e = np.concatenate([pv[w] - store[w]["true"] for w in ws])
        return float(np.sqrt(np.mean(e ** 2)))

    def apply_hedge(ws, q, wgt, thr=None):
        if q is None:
            return {w: store[w]["pred"] for w in ws}, thr
        if thr is None:
            thr = float(np.quantile([sig[w] for w in ws], q))
        out = {}
        for w in ws:
            p = store[w]["pred"]
            out[w] = (1 - wgt) * p + wgt * store[w]["anchor"] if sig[w] >= thr else p
        return out, thr

    oof = {}; picks = []
    for f in range(N_FOLDS):
        tr = [w for w in wells_l if fold_of[w] != f]
        va = [w for w in wells_l if fold_of[w] == f]
        best = (None, 0.0, np.inf, None)
        for q, wgt in GRID:
            pv, thr = apply_hedge(tr, q, wgt)
            r = pooled_pv(pv, tr)
            if r < best[2]:
                best = (q, wgt, r, thr)
        picks.append((best[0], best[1]))
        pv, _ = apply_hedge(va, best[0], best[1], thr=best[3])
        oof.update(pv)
    HEDGED = oof
    hedge_result = pooled_pv(oof, wells_l)
    print("per-fold picks (quantile, weight):", picks)
    print(f"hedged OOF pooled: {hedge_result:.3f}")
else:
    print("gate closed — hedge skipped, predictions unchanged")

gate closed — hedge skipped, predictions unchanged


## Final table + submission verdicts

In [6]:
def pooled_pv(pv):
    e = np.concatenate([pv[w] - store[w]["true"] for w in pv])
    return float(np.sqrt(np.mean(e ** 2)))
def perwell_pv(pv):
    return float(np.mean([rmse(pv[w], store[w]["true"]) for w in pv]))

base_pv = {w: store[w]["pred"] for w in store}
floor_pv = {w: np.full(store[w]["n_eval"], store[w]["anchor"]) for w in store}
print(f"{'stage':16s}{'pooled':>9s}{'per-well':>10s}")
print(f"{'floor':16s}{pooled_pv(floor_pv):9.3f}{perwell_pv(floor_pv):10.3f}")
print(f"{'pfx2_proj':16s}{pooled_pv(base_pv):9.3f}{perwell_pv(base_pv):10.3f}")
if GATE_OPEN:
    print(f"{'pfx2_proj+hedge':16s}{pooled_pv(HEDGED):9.3f}{perwell_pv(HEDGED):10.3f}")

print("\nSUBMISSION VERDICTS:")
print("  engine: pfx2 (dz_lik + lik_weight, GR path removed), projection ON")
if GATE_OPEN and pooled_pv(HEDGED) < pooled_pv(base_pv) - 1e-9:
    from collections import Counter
    q_pick = Counter(p[0] for p in picks).most_common(1)[0][0]
    w_pick = Counter(p[1] for p in picks).most_common(1)[0][0]
    print(f"  hedge: ON  signal={BEST_SIG}  quantile={q_pick}  weight={w_pick}"
          f"  (fold picks: {picks})")
else:
    print("  hedge: OFF")

stage              pooled  per-well
floor              18.009    13.873
pfx2_proj          14.854    11.209

SUBMISSION VERDICTS:
  engine: pfx2 (dz_lik + lik_weight, GR path removed), projection ON
  hedge: OFF


## Reading

1. `pfx2_proj` vs nb15's 15.020/11.298 on the same wells — the corrected config's
   true fleet number; below ~14.0 pooled / ~10.7 per-well is v4-class on this
   (harder) sample.
2. The diagnostic table — whether any confidence signal is real. A closed gate
   is a legitimate finding (it kills per-well hedging and points the tail work
   at the CSV-level blend instead).
3. The submission verdicts block — exactly what to carry into the submission
   notebook.